# 01 · Data Preparation

This minicourse builds a **multimodal survival-prediction pipeline** — combining whole-slide
histopathology images (WSI), multiparametric MRI (mpMRI), and structured clinical data — on the
[CHIMERA challenge](https://chimera.grand-challenge.org/) Task 1 dataset, predicting
**biochemical recurrence (BCR)** after prostatectomy.

A few design choices worth flagging up front:

1. **No layer-wise feature aggregation.** Some pipelines extract *every* intermediate transformer
   layer and aggregate them with a weighted grouping scheme. We skip that entirely and just take
   the final embedding — one vector per patch/scan, no extra aggregation step.
2. **[H-optimus-0](https://huggingface.co/bioptimus/H-optimus-0)** for WSI patch embeddings
   (notebook 02) — a ViT pathology foundation model loadable via `timm` (1536-dim embeddings,
   224×224 patches at 0.5 microns/pixel).
3. **Three interchangeable fusion strategies instead of one.** Rather than a single fixed fusion
   architecture, later in this course we implement **early**, **intermediate**, and **late**
   fusion side by side, so you can see how each one trades off differently.

The labels we prepare below support **two downstream tasks**, reusing the exact same
`BCR` / `time_to_follow-up/BCR` fields:

- **Classification** — predict the binary `BCR` event (recurred vs. not).
- **Survival analysis** — predict *time-to-event*, using `(time, event)` pairs with a
  censoring-aware loss.

**Course roadmap:**

| # | Notebook | Content |
|---|----------|---------|
| 01 | `01_data_preparation.ipynb` (this one) | Download data, WSI patch prep, clinical encoding, label prep + split |
| 02 | `02_feature_extraction.ipynb` | H-optimus-0 (WSI) and MRI-PTPCa (MRI) embeddings — single-pass, no layer aggregation |
| 03 | `03_fusion_models.ipynb` | Early / intermediate / late fusion architectures |
| 04 | `04_train.ipynb` | Train both tasks across all three fusion strategies and every CV fold |
| 05 | `05_inference_evaluation.ipynb` | Inference, aggregation, post-processing, evaluation & visualization |

**What this notebook covers:**

1. Download a subset of the CHIMERA Task 1 dataset
2. Explore the raw folder layout
3. Clinical data encoding (JSON → fixed-size vector)
4. WSI prep: tissue-filtered patch-coordinate extraction
5. Label preparation: parse BCR / time-to-event labels
6. Train/test split + stratified cross-validation folds

> **Note on MRI:** there is no MRI prep step here. The MRI-PTPCa pipeline's
> `load_and_preprocess_mri` already performs resampling, mask-based cropping, histogram
> equalization, and intensity normalization *inline*, as part of feature extraction — not as a
> separable step the way WSI patching is. So MRI handling is deferred in full to notebook 02.

## Setup

If you haven't already, run `00_environment_setup.ipynb` first to set up the `uv`-managed
environment and register its Jupyter kernel.

We keep this notebook self-contained: no project-specific code to import, just standard
libraries. (Later notebooks will import small model-architecture modules from `src/` — things
like the MRI-PTPCa CNN-ViT class — that genuinely don't belong copy-pasted into a notebook.
Everything in *this* notebook is plain data wrangling, so it all lives here.)

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import openslide
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split

# CHIMERA WSIs are large enough to trip PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

np.random.seed(42)

In [ ]:
# --- Project paths -----------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root

# Raw data, downloaded from the CHIMERA challenge task1 S3 bucket
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "task1"
CLINICAL_JSON_DIR = RAW_DATA_DIR / "clinical_data"
WSI_IMAGE_DIR = RAW_DATA_DIR / "pathology" / "images"
MRI_IMAGE_DIR = RAW_DATA_DIR / "radiology" / "images"  # untouched here, see notebook 02

# Artifacts this notebook produces
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
CLINICAL_EMBEDDING_DIR = PREPARED_DIR / "clinical_embeddings"
WSI_PATCH_MANIFEST_DIR = PREPARED_DIR / "wsi_patch_manifests"
LABELS_DIR = PROJECT_ROOT / "data" / "labels"

for d in (RAW_DATA_DIR, CLINICAL_EMBEDDING_DIR, WSI_PATCH_MANIFEST_DIR, LABELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Step 1 · Download the dataset

CHIMERA Task 1 is hosted on a public, no-sign-required S3 bucket:

```
s3://chimera-challenge/v2/task1/
├── clinical_data/
│   └── <patient_id>.json
├── pathology/
│   └── images/<patient_id>/<patient_id>_<scan_id>.tif (+ _tissue.tif mask)
└── radiology/
    └── images/<patient_id>/<patient_id>_<scan_id>_{t2w,adc,mask}.mha
```

The full training set is ~95 patients and several GB — more than we need for a tutorial. We'll
grab a small subset by name-filtering the `aws s3 sync` call. **To run the full pipeline, just set
`PATIENT_ID_SUBSET = None`.**

> **Already have the dataset locally** (e.g. on shared cluster storage)? Skip the download
> entirely — just symlink it into place and move on to Step 2:
> ```bash
> mkdir -p data/raw
> ln -s /path/to/existing/chimera/task1 data/raw/task1
> ```
> `RAW_DATA_DIR` below only cares that `clinical_data/`, `pathology/images/`, and
> `radiology/images/` exist under it — it doesn't matter whether they got there via `aws s3 sync`
> or a symlink.

In [ ]:
# A small, real subset of CHIMERA task1 patient IDs, for a fast tutorial download.
# Set this to None to download the entire training cohort instead (~95 patients).
PATIENT_ID_SUBSET = ["1003", "1010", "1011", "1021", "1025", "1026", "1028", "1030", "1031", "1035"]

CHIMERA_BUCKET_URI = "s3://chimera-challenge/v2/task1/"

# Flip to False once you're ready to actually pull data — this only *prints* the command by default
# so that re-running the notebook top-to-bottom never accidentally triggers a multi-GB download.
DRY_RUN = True

In [ ]:
if shutil.which("aws") is None:
    raise RuntimeError(
        "AWS CLI not found on PATH. Install it first, e.g. `pip install awscli`."
    )

cmd = ["aws", "s3", "sync", "--no-sign-request", CHIMERA_BUCKET_URI, str(RAW_DATA_DIR)]
if PATIENT_ID_SUBSET:
    cmd += ["--exclude", "*"]
    for pid in PATIENT_ID_SUBSET:
        cmd += ["--include", f"*{pid}*"]

print(" ".join(cmd))

if not DRY_RUN:
    subprocess.run(cmd, check=True)

## Step 2 · Explore the raw data

Let's look at what one patient's record actually contains before writing any preprocessing code.

In [ ]:
sample_patient_id = PATIENT_ID_SUBSET[0] if PATIENT_ID_SUBSET else next(
    p.stem for p in sorted(CLINICAL_JSON_DIR.glob("*.json"))
)

for label, path in [
    ("clinical", CLINICAL_JSON_DIR / f"{sample_patient_id}.json"),
    ("pathology", WSI_IMAGE_DIR / sample_patient_id),
    ("radiology", MRI_IMAGE_DIR / sample_patient_id),
]:
    print(f"--- {label}: {path} ---")
    if path.is_dir():
        for p in sorted(path.iterdir()):
            print(" ", p.name)
    elif path.exists():
        print(" ", path.name)
    else:
        print("  (not downloaded yet -- set DRY_RUN = False above and re-run Step 1)")

## Step 3 · Clinical data encoding

Each patient's clinical JSON is a mix of numeric, categorical, and free-text-ish fields (e.g.
`pT_stage: "pT2c"`, `earlier_therapy: "none"`). We turn each record into a **fixed-length float
vector** — same length and field order for every patient — so it can sit next to the WSI and MRI
embeddings as a third input modality:

- `pT_stage` → keep only the numeric part (`"pT2c"` → `2.0`)
- `earlier_therapy` (3-way categorical) → one-hot
- Binary clinical fields (lymph nodes, capsular penetration, LVI, surgical margins) also carry an
  "unknown" state → one-hot into 3 columns each (`_0`, `_1`, `_x`)
- Missing fields default to `0.0` rather than raising, so the encoding is robust to partially
  filled records

In [ ]:
# Fixed feature order: every patient gets a vector of this exact length and layout.
FEATURE_ORDER = [
    "age_at_prostatectomy", "primary_gleason", "secondary_gleason",
    "tertiary_gleason", "ISUP", "pre_operative_PSA",
    "pT_stage_numeric",
    "earlier_therapy_none", "earlier_therapy_cryo", "earlier_therapy_hormones",
    "positive_lymph_nodes_0", "positive_lymph_nodes_1", "positive_lymph_nodes_x",
    "capsular_penetration_0", "capsular_penetration_1", "capsular_penetration_x",
    "lymphovascular_invasion_0", "lymphovascular_invasion_1", "lymphovascular_invasion_x",
    "positive_surgical_margins_0", "positive_surgical_margins_1", "positive_surgical_margins_x",
]

ONE_HOT_BINARY_FIELDS = [
    "positive_lymph_nodes",
    "capsular_penetration",
    "lymphovascular_invasion",
    "positive_surgical_margins",
]


def _is_number(value) -> bool:
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def preprocess_clinical_data(data: dict) -> np.ndarray:
    """Convert one patient's clinical dict into a FEATURE_ORDER-shaped float32 vector."""
    item = dict(data)

    if "pT_stage" in item:
        digits = "".join(filter(str.isdigit, str(item["pT_stage"])))
        item["pT_stage_numeric"] = float(digits) if digits else 0.0

    if "earlier_therapy" in item:
        therapy = item["earlier_therapy"]
        item["earlier_therapy_none"] = float(therapy == "none")
        item["earlier_therapy_cryo"] = float(therapy == "radiotherapy + cryotherapy")
        item["earlier_therapy_hormones"] = float(therapy == "radiotherapy + hormones")

    for field in ONE_HOT_BINARY_FIELDS:
        if field in item:
            value = str(item[field]).lower().strip()
            is_0, is_1 = value in ("0", "0.0"), value in ("1", "1.0")
            item[f"{field}_0"] = float(is_0)
            item[f"{field}_1"] = float(is_1)
            item[f"{field}_x"] = float(not is_0 and not is_1)

    return np.array(
        [float(item.get(name, 0.0)) if _is_number(item.get(name, 0.0)) else 0.0
         for name in FEATURE_ORDER],
        dtype=np.float32,
    )

Try it on one patient first, before running it over the whole cohort:

In [ ]:
with open(CLINICAL_JSON_DIR / f"{sample_patient_id}.json") as f:
    sample_clinical = json.load(f)

sample_clinical

In [ ]:
sample_embedding = preprocess_clinical_data(sample_clinical)
print(sample_embedding.shape)
dict(zip(FEATURE_ORDER, sample_embedding))

Now run it over every patient and save one `.npy` embedding each:

In [ ]:
clinical_json_files = sorted(CLINICAL_JSON_DIR.glob("*.json"))
print(f"Found {len(clinical_json_files)} clinical records.")

for path in clinical_json_files:
    with open(path) as f:
        data = json.load(f)
    embedding = preprocess_clinical_data(data)
    np.save(CLINICAL_EMBEDDING_DIR / f"{path.stem}_embedding.npy", embedding)

print(f"Saved {len(clinical_json_files)} clinical embeddings to {CLINICAL_EMBEDDING_DIR}")

## Step 4 · WSI prep — tissue-filtered patch coordinates

WSIs are far too large to feed a model directly (gigapixel images), so they're processed as a grid
of small patches. We **split feature extraction into two stages**:

- **This notebook:** figure out *where* the tissue patches are, using the pyramid resolution grid
  and (if available) the CHIMERA-provided tissue mask, and save the coordinates.
- **Notebook 02:** load the H-optimus-0 foundation model and run it over each patch to get embeddings.

Decoupling them means patch geometry only needs to be computed once, and reusing/inspecting the
patch grid doesn't require reloading a foundation model.

In [ ]:
def extract_patch_coords(
    wsi_path: Path,
    tissue_mask_path: Path | None = None,
    patch_level: int = 0,
    patch_size: int = 224,
    patch_stride: int = 224,
    min_tissue_ratio: float = 0.2,
) -> np.ndarray:
    """Find top-left (x, y) coordinates of tissue-containing patches in a WSI."""
    slide = openslide.OpenSlide(str(wsi_path))
    try:
        width, height = slide.level_dimensions[patch_level]

        tissue_mask = None
        if tissue_mask_path is not None and tissue_mask_path.exists():
            tissue_mask = np.array(Image.open(tissue_mask_path).convert("L"))

        coords = []
        for y in range(0, height - patch_size + 1, patch_stride):
            for x in range(0, width - patch_size + 1, patch_stride):
                if tissue_mask is not None:
                    scale_x = tissue_mask.shape[1] / width
                    scale_y = tissue_mask.shape[0] / height
                    mask_x, mask_y = int(x * scale_x), int(y * scale_y)
                    mask_w = max(1, int(patch_size * scale_x))
                    mask_h = max(1, int(patch_size * scale_y))
                    region = tissue_mask[
                        mask_y : min(mask_y + mask_h, tissue_mask.shape[0]),
                        mask_x : min(mask_x + mask_w, tissue_mask.shape[1]),
                    ]
                    if region.size == 0 or (region > 0).mean() < min_tissue_ratio:
                        continue
                coords.append((x, y))
    finally:
        slide.close()

    return np.array(coords, dtype=np.int64)


# 224x224 matches H-optimus-0's expected input patch size (notebook 02).
PATCH_LEVEL = 0
PATCH_SIZE = 224
PATCH_STRIDE = 224
MIN_TISSUE_RATIO = 0.2

Try it on one WSI and visualize the resulting patch grid over a thumbnail:

In [ ]:
sample_wsi_dir = WSI_IMAGE_DIR / sample_patient_id
sample_wsi_path = sorted(p for p in sample_wsi_dir.glob("*.tif") if not p.stem.endswith("_tissue"))[0]
sample_wsi_id = sample_wsi_path.stem
sample_tissue_mask_path = sample_wsi_path.with_name(f"{sample_wsi_id}_tissue.tif")

sample_coords = extract_patch_coords(
    sample_wsi_path,
    tissue_mask_path=sample_tissue_mask_path if sample_tissue_mask_path.exists() else None,
    patch_level=PATCH_LEVEL,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
    min_tissue_ratio=MIN_TISSUE_RATIO,
)

print(f"{sample_wsi_id}: {len(sample_coords)} tissue patches")

In [ ]:
slide = openslide.OpenSlide(str(sample_wsi_path))
thumbnail = slide.get_thumbnail((1024, 1024))
scale = thumbnail.size[0] / slide.level_dimensions[PATCH_LEVEL][0]
slide.close()

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(thumbnail)
for x, y in sample_coords[:: max(1, len(sample_coords) // 2000)]:  # subsample for a readable plot
    ax.add_patch(mpatches.Rectangle(
        (x * scale, y * scale), PATCH_SIZE * scale, PATCH_SIZE * scale,
        linewidth=0.5, edgecolor="lime", facecolor="none",
    ))
ax.set_title(f"{sample_wsi_id} -- {len(sample_coords)} tissue patches")
ax.axis("off")
plt.show()

Now run it over every WSI and save each one's coordinates as a small manifest file:

In [ ]:
wsi_paths = sorted(p for p in WSI_IMAGE_DIR.glob("*/*.tif") if not p.stem.endswith("_tissue"))
print(f"Found {len(wsi_paths)} WSIs.")

for wsi_path in wsi_paths:
    wsi_id = wsi_path.stem
    tissue_mask_path = wsi_path.with_name(f"{wsi_id}_tissue.tif")

    coords = extract_patch_coords(
        wsi_path,
        tissue_mask_path=tissue_mask_path if tissue_mask_path.exists() else None,
        patch_level=PATCH_LEVEL,
        patch_size=PATCH_SIZE,
        patch_stride=PATCH_STRIDE,
        min_tissue_ratio=MIN_TISSUE_RATIO,
    )

    np.savez(
        WSI_PATCH_MANIFEST_DIR / f"{wsi_id}_patches.npz",
        coords=coords, patch_size=PATCH_SIZE, patch_level=PATCH_LEVEL,
    )

print(f"Saved {len(wsi_paths)} patch manifests to {WSI_PATCH_MANIFEST_DIR}")

## Step 5 · Label preparation

The same clinical JSON files also carry the outcome we're predicting: `BCR` (did biochemical
recurrence occur?) and `time_to_follow-up/BCR` (months of follow-up, or months to recurrence).
These two fields are deliberately excluded from `FEATURE_ORDER` above — they're labels, not
predictors.

- `BCR` alone is the label for the **classification** task.
- `(time_to_follow-up/BCR, BCR)` together is the label for the **survival** task — `BCR` doubles
  as the censoring indicator (0 = still censored at last follow-up, 1 = event observed).

In [ ]:
TIME_FIELD = "time_to_follow-up/BCR"
EVENT_FIELD = "BCR"

rows = []
for path in clinical_json_files:
    with open(path) as f:
        data = json.load(f)
    if TIME_FIELD not in data or EVENT_FIELD not in data:
        available = ", ".join(sorted(data.keys()))
        raise KeyError(
            f"'{TIME_FIELD}' / '{EVENT_FIELD}' not found in {path.name}. "
            f"Available fields: {available}"
        )
    rows.append({
        "patient_id": int(path.stem),
        TIME_FIELD: float(data[TIME_FIELD]),
        EVENT_FIELD: float(data[EVENT_FIELD]),
    })

labels_df = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
print(f"BCR event rate: {labels_df[EVENT_FIELD].mean():.2%}")
labels_df.head()

## Step 6 · Train/test split + cross-validation folds

BCR cohorts are small and imbalanced (most patients are censored, only a minority recur), so we
**stratify on the event indicator** at every split — otherwise a random split could easily produce
a test set (or a fold) with almost no observed events, making the C-index or AUC on it meaningless.

We hold out a test set first, then assign stratified K-fold splits *within* the training set for
cross-validation, tracked via a `fold` column on the training labels.

In [ ]:
TEST_SIZE = 0.2
N_FOLDS = 5
SEED = 42

# With a very small demo subset (Step 1), there may not be enough events per class for 5 folds --
# shrink N_FOLDS to fit however many are available.
n_events = int(labels_df[EVENT_FIELD].sum())
n_non_events = len(labels_df) - n_events
n_folds_effective = max(2, min(N_FOLDS, n_events, n_non_events)) if min(n_events, n_non_events) > 0 else 1
if n_folds_effective != N_FOLDS:
    print(f"Only {n_events} events / {n_non_events} non-events available -- using "
          f"{n_folds_effective} folds instead of {N_FOLDS}. Download more patients for a "
          "realistic cross-validation setup.")

train_df, test_df = train_test_split(
    labels_df, test_size=TEST_SIZE, stratify=labels_df[EVENT_FIELD], random_state=SEED,
)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df["fold"] = -1
if n_folds_effective >= 2:
    skf = StratifiedKFold(n_splits=n_folds_effective, shuffle=True, random_state=SEED)
    for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df[EVENT_FIELD])):
        train_df.loc[train_df.index[val_idx], "fold"] = fold_idx
else:
    train_df["fold"] = 0  # too few events to stratify -- single fold as a placeholder

test_df["fold"] = -1  # held out; not used for cross-validation

print(f"Train: {len(train_df)} patients across {train_df['fold'].nunique()} folds")
print(f"Test:  {len(test_df)} patients (held out)")
train_df["fold"].value_counts().sort_index()

In [ ]:
train_labels_path = LABELS_DIR / "train_labels.csv"
test_labels_path = LABELS_DIR / "test_labels.csv"

train_df.to_csv(train_labels_path, index=False)
test_df.to_csv(test_labels_path, index=False)

print(f"Saved {train_labels_path}")
print(f"Saved {test_labels_path}")

## Summary

This notebook produced:

| Artifact | Location | Used by |
|---|---|---|
| Clinical embeddings | `data/prepared/clinical_embeddings/*_embedding.npy` | Notebook 03/04 (clinical modality input) |
| WSI patch manifests | `data/prepared/wsi_patch_manifests/*_patches.npz` | Notebook 02 (WSI feature extraction) |
| Train labels + CV folds | `data/labels/train_labels.csv` | Notebook 04 (cross-validation training) |
| Held-out test labels | `data/labels/test_labels.csv` | Notebook 04 (final evaluation) |

**No MRI artifacts were produced here** — as noted at the top, MRI-PTPCa's preprocessing
(resampling, mask-cropping, histogram equalization, normalization) happens inline as part of
feature extraction, so there's nothing to precompute separately.

**Next up — `02_feature_extraction.ipynb`:** load H-optimus-0 and run it over the WSI patch
coordinates from Step 4 (final-layer embedding only, no layer-wise aggregation), and run the full
MRI-PTPCa pipeline (prep + extract in one) over the radiology images.